# Claude Managed Agents로 SRE incident 대응 agent 만들기

## 소개

새벽 3시에 production alert가 발생하면 누군가는 log를 가져오고, 올바른 runbook을 찾고, 잘못된 설정을 추적하고, PR을 열고, 승인을 받아야 합니다. 적절한 context가 있고 사람이 최종 판단을 한다면, agent가 초기 작업을 대신 수행하여 당신이 키보드 앞에 앉을 때 검토할 수정안을 준비해 둘 수 있습니다.

[Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview)는 이를 쉽게 구축할 수 있는 확장 가능한 infrastructure, sandboxing, security 요소를 제공합니다. 이 tutorial에서는 이들을 연결합니다.

- 모의 **PagerDuty webhook**이 API 호출 한 번으로 Claude Managed Agent를 실행합니다.
- **Skill**이 agent에 팀의 runbook 규칙을 가르쳐 어디를 볼지 알게 합니다.
- built-in `bash`/`read`/`edit` tool로 sandbox에서 log와 infrastructure 코드를 조사합니다.
- **Custom tool**로 pull request를 열고 merge 전에 사람에게 승인을 요청합니다. 코드는 해당 호출을 처리하므로 "PR 열기"가 실제로 무엇을 할지 직접 정합니다.
- **Anthropic Console**이 모든 단계를 자동 기록하여 완전한 observability를 제공합니다.

아래 모든 내용은 `ANTHROPIC_API_KEY`만으로 실행됩니다. PagerDuty, GitHub, Datadog은 로컬 fixture로 mock 처리하여 Managed Agents 요소에 집중할 수 있습니다. 마지막 절은 각 mock을 실제 서비스로 바꾸는 방법을 보여 줍니다.

### 배울 내용

- Skill을 업로드하여 Claude Managed Agent에 연결하기
- built-in toolset과 애플리케이션에서 처리하는 custom tool 혼합하기
- webhook payload로 세션 시작하기
- 파괴적인 작업에 사람의 승인 조건 두기
- Console에서 전체 세션 trace 읽기

### 사전 요구 사항

environment에 `ANTHROPIC_API_KEY`를 설정한 뒤 dependency를 설치하세요.


In [1]:
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
import json
import os
import time
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from utilities import wait_for_idle_status

load_dotenv()
client = Anthropic()
MODEL = os.getenv("COOKBOOK_MODEL", "claude-opus-4-6")
FIXTURE = Path("example_data/sre")

## 1. runbook skill 업로드하기

[**Skill**](https://platform.claude.com/docs/en/managed-agents/skills)은 플랫폼이 progressive disclosure로 agent context에 mount하는 작은 filesystem bundle입니다. agent는 먼저 한 줄 설명을 보고 관련될 때만 본문을 읽습니다. system prompt에 두지 말아야 할 팀 규칙을 넣기에 적합합니다.

아래 예제 skill은 실제 팀 playbook처럼 하나의 규칙, *infrastructure를 건드리기 전에 runbook을 확인하기*를 담습니다. Skills API로 한 번 업로드하고 필요한 모든 agent에서 ID로 참조합니다.


In [3]:
# 실제 skill은 보통 디스크의 폴더(SKILL.md와 helper script나 reference 문서)이며
# zip으로 압축하여 업로드합니다. 이 tutorial에서는 SKILL.md가
# 충분히 작아 내부에 유지합니다.
RUNBOOK_SKILL = """\
---
name: incident-runbooks
description: How to triage production incidents using the team runbooks.
---

# Incident runbooks

When an alert references a service, locate that service's recent logs
and identify the failure signature (the repeating error class, exit
code, or status pattern).

Consult the team runbooks before proposing any fix. Runbooks are
organised by failure signature — for example `oom.md`, `5xx.md`,
`latency.md`. Each one lists the triage steps for that class of
failure and the configuration that usually needs to change.

Any fix to infrastructure code must be opened as a pull request that
cites the runbook you followed. Do not patch live resources directly.
"""

skill = client.beta.skills.create(
    display_title="incident-runbooks",
    files=[("incident-runbooks/SKILL.md", RUNBOOK_SKILL.encode(), "text/markdown")],
)
print(f"skill: {skill.id} (version {skill.latest_version})")

skill: skill_01WPWHALbtEVBUWG6mHa7Tna (version 1775588716519983)


## 2. agent 생성하기

agent의 `tools` 목록은 세 종류의 기능을 결합합니다.

- [`agent_toolset_20260401`](https://platform.claude.com/docs/en/managed-agents/tools) — sandbox *안에서* 실행하는 built-in `bash`, `read`, `grep`, `edit`, … tool입니다. agent는 조사에 이를 사용합니다.
- 1단계의 runbook **skill**.
- **custom tool** 세 개 — `open_pull_request`, `request_approval`, `merge_pull_request`입니다. agent가 호출하지만 *애플리케이션*이 실행합니다. agent가 sandbox 밖의 시스템에 접근하는 방법이자 사람이 참여하게 하는 방법입니다.

system prompt에는 persona와 workflow만 있습니다. alert 자체는 첫 user event로 도착하므로 같은 agent가 모든 incident를 처리합니다.


In [4]:
SRE_SYSTEM_PROMPT = """\
You are an on-call SRE agent. Each user message is a PagerDuty alert
payload. Triage it to root cause and ship the minimal safe fix.

The session workspace contains the recent logs, the infrastructure
repo, and the team runbooks for the alerting service. Explore it to
find what you need.

Workflow for every alert:
1. Read the logs and identify the failure signature.
2. Find the root cause in the infrastructure repo, save a copy of the
   original file, edit it in place, then produce a unified diff with
   `diff -u`.
3. open_pull_request(title, body, diff) with the fix.
4. request_approval(summary) and wait for the human's decision.
5. Only if the result is "approved", merge_pull_request(pr_number).
   Otherwise stop and report.

Never call merge_pull_request unless request_approval returned
"approved". Keep the fix minimal — do not refactor unrelated config.
"""

agent = client.beta.agents.create(
    name="cookbook-sre-responder",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/sre-incident-responder"},
    model=MODEL,
    system=SRE_SYSTEM_PROMPT,
    skills=[{"type": "custom", "skill_id": skill.id, "version": skill.latest_version}],
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        },
        {
            "type": "custom",
            "name": "open_pull_request",
            "description": "Open a pull request against the infra repo with the proposed fix.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "body": {"type": "string"},
                    "diff": {"type": "string", "description": "Unified diff of the change."},
                },
                "required": ["title", "body", "diff"],
            },
        },
        {
            "type": "custom",
            "name": "request_approval",
            "description": "Ask the on-call human to approve the proposed PR before merging.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                },
                "required": ["summary"],
            },
        },
        {
            "type": "custom",
            "name": "merge_pull_request",
            "description": "Merge an approved pull request.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "pr_number": {"type": "integer"},
                },
                "required": ["pr_number"],
            },
        },
    ],
)
print(f"agent: {agent.id} v{agent.version}")

agent: agent_011CZpw3Y76Vu4t2j2QEosVa v1


## 3. environment 생성하고 데이터 mount하기

agent가 조사하려면 작업 공간에 세 가지가 필요합니다. 최근 서비스 log, infrastructure repo, 팀 runbook입니다. 각각 Files API로 업로드하고 `resources`에 나열하여 모든 세션에 system prompt가 기대하는 경로로 mount되게 합니다. agent는 자체 filesystem만 필요하므로 `limited` networking cloud environment면 충분합니다.

이 notebook을 `ANTHROPIC_API_KEY`만으로 실행할 수 있도록 infra "repo"는 너무 낮은 `memory: 128Mi` 한도가 있는 단일 manifest입니다. production에서는 업로드 대신 실제 repo를 sandbox에 바로 clone하는 `github_repository` resource를 사용합니다.

```python
{
    "type": "github_repository",
    "url": "https://github.com/your-org/infra",
    "authorization_token": os.environ["GITHUB_TOKEN"],
    "checkout": {"type": "branch", "name": "main"},
    "mount_path": "infra",
}
```


In [5]:
env = client.beta.environments.create(
    name="cookbook-sre-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)


def upload(path: Path, mime: str) -> str:
    with path.open("rb") as f:
        return client.beta.files.upload(file=(path.name, f, mime)).id


log_id = upload(FIXTURE / "logs/checkout-svc.log", "text/plain")
manifest_id = upload(FIXTURE / "infra/k8s/checkout-deploy.yaml", "text/yaml")
runbook_id = upload(FIXTURE / "runbooks/oom.md", "text/markdown")

RESOURCES = [
    {"type": "file", "file_id": log_id, "mount_path": "logs/checkout-svc.log"},
    {"type": "file", "file_id": manifest_id, "mount_path": "infra/k8s/checkout-deploy.yaml"},
    {"type": "file", "file_id": runbook_id, "mount_path": "runbooks/oom.md"},
]
print(f"environment: {env.id}")

environment: env_01R6hmJkd6BhpPotXnoC7rqU


## 4. incident alert 처리하기

아래 handler는 deploy할 하나의 함수입니다. incident가 발생하면 alerting 시스템이 호출하는 Flask 또는 FastAPI route입니다. agent와 environment를 참조하는 세션을 만들고, 데이터를 mount하고, alert JSON을 첫 `user.message` event로 보냅니다. 예제는 [PagerDuty V3 webhook](https://developer.pagerduty.com/docs/webhooks-overview) payload를 사용하지만 JSON을 POST할 수 있는 모든 알림 시스템이 같은 방식으로 작동합니다. 여기서는 fixture로 handler를 직접 호출합니다.


In [6]:
def handle_pagerduty_webhook(payload: dict) -> str:
    incident = payload["event"]["data"]
    session = client.beta.sessions.create(
        environment_id=env.id,
        agent={"type": "agent", "id": agent.id, "version": agent.version},
        resources=RESOURCES,
        title=f"[{incident['service']['summary']}] {incident['title']}",
    )
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [{"type": "text", "text": json.dumps(payload, indent=2)}],
            }
        ],
    )
    return session.id


with (FIXTURE / "alert.json").open() as f:
    alert = json.load(f)

session_id = handle_pagerduty_webhook(alert)
print(f"session: {session_id}")

session: sesn_011CZpw3gtC691y7qmaLNmLM


## 5. agent의 custom tool 호출 처리하기

built-in tool과 custom tool이 함께 작동하는 곳입니다. agent의 `read`/`bash`/`edit` 호출은 container에서 실행되고 event log에 `agent.tool_use`로 나타납니다. 이것이 조사 과정이며 출력만 하면 됩니다. 하지만 agent가 custom tool 중 하나를 호출하면 세션은 `stop_reason.type == "requires_action"`과 함께 `idle`이 되고 *애플리케이션*이 `user.custom_tool_result`로 응답하기를 기다립니다.

아래 루프는 `events.list`를 polling하고 로컬 목록에 기록하여 `open_pull_request` 및 `merge_pull_request`에 즉시 응답합니다. 이것이 GitHub mock입니다. 하지만 `request_approval`이 도착하면 사람이 필요하므로 **반환합니다**.

production에서 "사람이 필요함"은 보통 *Slack에 게시하기*를 뜻합니다. agent 요약을 **Approve** 버튼과 함께 on-call channel에 올리고 누군가 클릭하면 결과를 돌려보냅니다. [`slack_data_bot` cookbook](slack_data_bot.ipynb)이 Bolt 연결을 보여 줍니다. 여기서는 notebook의 자체 완결성을 위해 다음 셀에서 직접 승인합니다.


In [7]:
prs: list[dict] = []
pending_approvals: list[dict] = []
seen_events: set[str] = set()


def handle_custom_tool(name: str, args: dict) -> dict:
    if name == "open_pull_request":
        n = len(prs) + 1
        prs.append({"number": n, "merged": False, **args})
        print(f"\n── PR #{n}: {args['title']} ──")
        return {"pr_number": n, "url": f"mock://infra/pull/{n}"}
    if name == "merge_pull_request":
        prs[args["pr_number"] - 1]["merged"] = True
        return {"merged": True}
    raise ValueError(f"unhandled tool {name}")


def run_until_approval_or_end(session_id: str) -> str | None:
    """세션 event log를 polling하며 custom tool을 처리합니다.
    request_approval 호출이 도착하면 event_id를 반환하여 caller가 응답하게 하고,
    agent가 turn을 끝내면 None을 반환합니다."""
    custom_calls: dict[str, object] = {}
    responded: set[str] = set()
    while True:
        idle_stop = None
        for ev in client.beta.sessions.events.list(session_id):
            if ev.id in seen_events:
                continue
            seen_events.add(ev.id)
            if ev.type == "agent.message":
                for block in ev.content:
                    if block.type == "text":
                        print(block.text, end="")
            elif ev.type == "agent.tool_use":
                print(f"\n  [{ev.name}]")
            elif ev.type == "agent.custom_tool_use":
                custom_calls[ev.id] = ev
                print(f"\n→ {ev.name}")
            elif ev.type == "session.status_idle":
                idle_stop = ev.stop_reason
            elif ev.type == "session.status_terminated":
                return None
        if idle_stop is None:
            time.sleep(1.0)
            continue
        if idle_stop.type == "end_turn":
            return None
        if idle_stop.type == "requires_action":
            for event_id in idle_stop.event_ids:
                if event_id in responded:
                    continue
                call = custom_calls[event_id]
                if call.name == "request_approval":
                    pending_approvals.append({"event_id": event_id, **call.input})
                    return event_id
                result = handle_custom_tool(call.name, call.input)
                client.beta.sessions.events.send(
                    session_id,
                    events=[
                        {
                            "type": "user.custom_tool_result",
                            "custom_tool_use_id": event_id,
                            "content": [{"type": "text", "text": json.dumps(result)}],
                        }
                    ],
                )
                responded.add(event_id)
        time.sleep(1.0)


approval_event_id = run_until_approval_or_end(session_id)

I'll help you triage this production incident. The alert shows checkout-svc pods are crash-looping with 7 restarts in 5 minutes. Let me start by exploring the workspace to find logs and the infrastructure configuration.
  [bash]
Let me check the session directory for logs and infrastructure files:
  [bash]
Perfect! Let me examine the logs first to identify the failure signature:
  [read]
Perfect! I've identified the failure signature. The logs show a clear **OutOfMemoryError** pattern:

**Failure Signature:**
- Service starts and warms pricing cache (14,092 entries)
- Heap memory grows rapidly: 101MB → 118MB → 121MB (against 128MB limit)
- `pricing.recompute` function attempts allocation during garbage collection pause (412ms GC pause observed)
- Container gets OOMKilled (exit 137) after ~2 minutes
- Service restarts and immediately repeats the cycle

Now let me examine the deployment configuration to identify the root cause:
  [read]
**Root Cause Analysis:**

The deployment has memory

agent는 log를 읽고, skill을 통해 `OOMKilled` 특징을 `runbooks/oom.md`와 연결하고, `infra/k8s/checkout-deploy.yaml`의 128Mi 메모리 한도를 찾고 편집한 뒤 PR을 열었으며, 이제 당신을 기다립니다. `#oncall` Slack channel에 도착할 message는 다음과 같습니다.


In [8]:
pr = prs[0]
print(pr["body"], "\n")
print(pr["diff"])
print("\n" + "─" * 60)
print("APPROVAL REQUESTED:", pending_approvals[0]["summary"])

## Issue
checkout-svc pods are in a CrashLoopBackOff state due to OutOfMemoryError. The service consistently crashes after ~2 minutes with 7 restarts in the last 5 minutes.

## Root Cause
The deployment had memory limits set to 128Mi, which is insufficient for the pricing cache operation:
- Pricing cache warms with 14,092 entries during startup
- Heap pressure builds to 118-121MB (92-94% of limit) within 90 seconds
- pricing.recompute fails to allocate 8MB, causing OOMKilled (exit 137)
- Service restarts and repeats the cycle

## Fix
Increase memory allocation to provide adequate headroom:
- **Memory request:** 128Mi → 256Mi
- **Memory limit:** 128Mi → 512Mi

This provides 4x headroom for the pricing cache and normal operations while remaining resource-efficient (512Mi limit is standard for Java/similar workloads with caching).

## Verification
The fix addresses the immediate OOMKilled pattern in logs and aligns memory resources with the actual cache size and operational requirements. 

## 6. 승인하고 agent가 merge하게 하기

`request_approval` 결과로 `"approved"`를 돌려보냅니다. agent가 재개하여 `merge_pull_request`를 호출하고 turn을 끝냅니다. Slack version에서는 이 전송이 버튼 클릭 handler에서 이루어지며 payload는 동일합니다.


In [9]:
client.beta.sessions.events.send(
    session_id,
    events=[
        {
            "type": "user.custom_tool_result",
            "custom_tool_use_id": approval_event_id,
            "content": [{"type": "text", "text": json.dumps({"decision": "approved"})}],
        }
    ],
)

run_until_approval_or_end(session_id)
print(f"\n\nPR #{pr['number']} merged: {prs[0]['merged']}")

Great! The fix has been approved. Now merging:
→ merge_pull_request
## ✅ Incident Resolved

**Summary:**
- **Status:** MERGED (PR #1)
- **Failure:** checkout-svc crash-loop with OOMKilled (exit 137)
- **Root Cause:** Memory limit of 128Mi was insufficient for pricing cache (14,092 entries)
- **Fix Applied:** 
  - Memory request: 128Mi → 256Mi
  - Memory limit: 128Mi → 512Mi

**What Happened:**
1. Service loaded 14k+ pricing cache entries during startup
2. Heap grew to 118-121MB within ~2 minutes (92-94% of 128Mi limit)
3. pricing.recompute failed to allocate 8MB, triggering OutOfMemoryError
4. Container was OOMKilled and restarted, repeating the cycle

**Expected Outcome:**
With 512Mi limit and 256Mi request, the service will have sufficient memory headroom for:
- Pricing cache operations
- Normal request processing
- JVM garbage collection pauses
- No more CrashLoopBackOff

The deployment update will trigger a rolling restart of the 3 replicas, allowing pods to spawn with the new memo

## 7. Console에서 실행 검토하기

조사가 Managed Agents 세션으로 실행되었으므로 위의 모든 단계 — 파일 읽기, `bash` diff, manifest 편집, custom tool 호출 세 개, 보낸 승인 — 가 세션 event로 보존됩니다. 추가 instrumentation 없이 전체 audit trail을 확인하려면 [Console](https://platform.claude.com/)의 **Managed Agents → Sessions**에서 여세요.

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/sre/console_session.png" alt="incident 대응 실행의 Console 세션 보기" width="700" />

### 정리하기

세션과 생성한 resource를 archive하세요.


In [10]:
wait_for_idle_status(client, session_id)
client.beta.sessions.archive(session_id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.skills.versions.delete(skill.latest_version, skill_id=skill.id)
client.beta.skills.delete(skill.id)
print("archived")

archived


## 다음 단계: production 연결

세 가지 교체로 notebook에서 on-call로 옮길 수 있습니다.

**Slack에서 승인하기.** `request_approval`이 도착하면 Block Kit 버튼과 함께 on-call channel에 게시하고 action handler에서 `user.custom_tool_result`를 돌려보냅니다. [`slack_data_bot` cookbook](slack_data_bot.ipynb)이 Bolt app 설정을 다룹니다. 승인에 특화된 부분은 작습니다.

```python
def post_for_approval(session_id, event_id, summary):
    slack.client.chat_postMessage(
        channel=ONCALL_CHANNEL,
        text=summary,
        blocks=[
            {"type": "section", "text": {"type": "mrkdwn", "text": summary}},
            {"type": "actions", "elements": [
                {"type": "button", "text": {"type": "plain_text", "text": "Approve"},
                 "action_id": "approve", "value": f"{session_id}:{event_id}"},
                {"type": "button", "text": {"type": "plain_text", "text": "Reject"},
                 "action_id": "reject", "value": f"{session_id}:{event_id}"},
            ]},
        ],
    )

@slack.action("approve")
def on_approve(ack, body):
    ack()
    session_id, event_id = body["actions"][0]["value"].split(":")
    client.beta.sessions.events.send(
        session_id,
        events=[{"type": "user.custom_tool_result",
                 "custom_tool_use_id": event_id,
                 "content": [{"type": "text",
                              "text": json.dumps({"decision": "approved"})}]}],
    )
```

polling 루프를 완전히 없애려면 `session.requires_action`에 Console webhook을 등록하세요. agent가 일시 정지하는 순간 플랫폼이 endpoint를 호출하며 거기서 Slack에 게시합니다.

**mock 대신 GitHub.** `open_pull_request` / `merge_pull_request` custom tool을 제거하고 대신 GitHub MCP server를 제공하세요. token은 vault에 저장하여 코드에 나타나지 않게 합니다. [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb)는 사용자별 credential을 설명합니다.

```python
agent = client.beta.agents.create(
    ...,
    mcp_servers=[{"type": "url", "name": "github",
                  "url": "https://api.githubcopilot.com/mcp/"}],
    tools=[{"type": "agent_toolset_20260401", ...},
           {"type": "mcp_toolset", "server_name": "github"},
           {"type": "custom", "name": "request_approval", ...}],
)
session = client.beta.sessions.create(..., vault_ids=[github_vault.id])
```

**fixture 대신 실시간 log.** environment config로 `DD_API_KEY` / `DD_APP_KEY`를 전달하고, mount된 파일을 읽는 대신 agent가 `bash`에서 Datadog Logs API를 `curl`하게 합니다.

> 같은 문제를 hosted Managed Agents runtime 대신 로컬 Agent SDK로 해결한 [Agent SDK site-reliability agent](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/03_The_site_reliability_agent.ipynb)도 참고하세요.


## 배운 내용

- 모든 외부 event에서 세션 시작하기 — PagerDuty webhook의 API 호출 한 번이 전체 실행을 시작했습니다.
- **Skill**을 연결하여 agent에 팀 규칙 제공하기.
- **resource**로 데이터 mount하기: 코드에는 `github_repository`, log와 runbook에는 `file`.
- **custom tool**로 앱을 다시 호출하고 `requires_action`을 통해 사람 승인 조건 두기.
- **Console** 세션 보기에서 추가 작업 없이 전체 audit trail 얻기.

mock을 GitHub MCP, Slack 승인 버튼, 실시간 log로 바꾸면 on-call 준비가 됩니다.
